In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU name: Tesla T4


In [ ]:
!mkdir -p experiments/02_fasttext

In [ ]:
from google.colab import files

uploaded = files.upload()

import os
for filename in uploaded.keys():
    if 'corpus_clean' in filename:
        os.rename(filename, 'experiments/02_fasttext/corpus_clean.json')
        print(f"✅ Arquivo {filename} movido para experiments/02_fasttext/corpus_clean.json")

Saving corpus_clean.json to corpus_clean.json
✅ Arquivo corpus_clean.json movido para experiments/02_fasttext/corpus_clean.json


In [ ]:
%%writefile finetune_colab.py
import json
import random
import numpy as np
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

# --- Paths (usando corpus limpo) ---
CORPUS_PATH = "experiments/02_fasttext/corpus_clean.json"
OUTPUT_DIR = "xlmr_finetuned_clean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Reproducibility ---
rng = random.Random(42)
np.random.seed(42)
torch.manual_seed(42)

# --- Load and split data ---
print(f"\nLoading corpus from: {CORPUS_PATH}")
with open(CORPUS_PATH, encoding="utf-8") as f:
    data = json.load(f)

print(f"Total pairs: {len(data)} (bad pairs removed)")

rng.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
test_data = data[split_idx:]

print(f"Train pairs: {len(train_data)}")
print(f"Test pairs:  {len(test_data)}")

# --- Dataset ---
class ParallelDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]['pt'], self.pairs[idx]['nhe']

def collate_fn(batch, tokenizer):
    pt_texts = [item[0] for item in batch]
    nhe_texts = [item[1] for item in batch]
    pt_enc = tokenizer(pt_texts, padding=True, truncation=True,
                       max_length=128, return_tensors='pt')
    nhe_enc = tokenizer(nhe_texts, padding=True, truncation=True,
                        max_length=128, return_tensors='pt')
    return pt_enc, nhe_enc

# --- Model & Tokenizer ---
print("\nLoading XLM-R model...")
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

BATCH_SIZE = 16
dataset = ParallelDataset(train_data)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=lambda b: collate_fn(b, tokenizer))

optimizer = AdamW(model.parameters(), lr=2e-5)

# --- Mean Pooling ---
def mean_pool(model_output, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (model_output.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)

# --- Training ---
EPOCHS = 3
print(f"\nStarting training for {EPOCHS} epochs...")
print("="*60)

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for step, (pt_enc, nhe_enc) in enumerate(loader):
        pt_enc = {k: v.to(device) for k, v in pt_enc.items()}
        nhe_enc = {k: v.to(device) for k, v in nhe_enc.items()}

        pt_out = model(**pt_enc)
        nhe_out = model(**nhe_enc)

        pt_embs = mean_pool(pt_out, pt_enc['attention_mask'])
        nhe_embs = mean_pool(nhe_out, nhe_enc['attention_mask'])

        pt_embs = F.normalize(pt_embs, p=2, dim=1)
        nhe_embs = F.normalize(nhe_embs, p=2, dim=1)

        scores = torch.matmul(pt_embs, nhe_embs.T) * 20
        labels = torch.arange(len(pt_embs), device=device)
        loss = F.cross_entropy(scores, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if step % 50 == 0:
            print(f"Epoch {epoch+1}, step {step}: loss = {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} complete. Average loss: {avg_loss:.4f}")

# --- Save model ---
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Model saved to {OUTPUT_DIR}")

Overwriting finetune_colab.py


In [ ]:
!pip install torch transformers

In [ ]:
%%writefile eval_colab.py
"""
Avaliação completa do XLM-R fine-tuned com corpus limpo
Inclui métricas e análise qualitativa
"""

import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

CORPUS_PATH = "experiments/02_fasttext/corpus_clean.json"
MODEL_PATH = "xlmr_finetuned_clean"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("="*60)
print("AVALIAÇÃO XLM-R FINE-TUNED (CORPUS LIMPO)")
print("="*60)

# Load data
print("\n1. Carregando corpus...")
with open(CORPUS_PATH, encoding="utf-8") as f:
    data = json.load(f)

print(f"   Total pairs: {len(data)} (bad pairs removed)")

# Split (mesmo seed do treino)
random.seed(42)
np.random.seed(42)
random.shuffle(data)
split_idx = int(0.8 * len(data))
test_data = data[split_idx:]

print(f"   Test pairs: {len(test_data)}")

# Carregar modelo fine-tuned
print("\n2. Carregando modelo fine-tuned...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModel.from_pretrained(MODEL_PATH).to(device)
model.eval()

def embed_texts(texts, batch_size=32):
    """Gera embeddings."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=128, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**encoded)
        mask = encoded['attention_mask'].unsqueeze(-1).float()
        embs = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        embs = F.normalize(embs, p=2, dim=1)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

print("\n3. Computando embeddings...")
print("   PT:", end=' ')
pt_embs = embed_texts([p['pt'] for p in test_data])
print(f" {pt_embs.shape}")
print("   NHE:", end=' ')
nhe_embs = embed_texts([p['nhe'] for p in test_data])
print(f" {nhe_embs.shape}")

# ============================================================
# AVALIAÇÃO QUANTITATIVA
# ============================================================
print("\n4. Avaliação quantitativa...")

p1 = p5 = p10 = mrr = 0
n = len(pt_embs)

for i, pt in enumerate(pt_embs):
    sims = np.dot(nhe_embs, pt)
    rank = np.argsort(-sims)
    pos = np.where(rank == i)[0][0] + 1
    mrr += 1.0 / pos
    if pos == 1: p1 += 1
    if pos <= 5: p5 += 1
    if pos <= 10: p10 += 1

print(f"\n{'='*60}")
print(f"RESULTADOS QUANTITATIVOS")
print(f"{'='*60}")
print(f"Test pairs: {n}")
print(f"Precision@1:  {p1/n:.4f} ({p1}/{n})")
print(f"Precision@5:  {p5/n:.4f} ({p5}/{n})")
print(f"Precision@10: {p10/n:.4f} ({p10}/{n})")
print(f"MRR:          {mrr/n:.4f}")

# ============================================================
# ANÁLISE QUALITATIVA
# ============================================================
print(f"\n{'='*60}")
print(f"ANÁLISE QUALITATIVA")
print(f"{'='*60}")

# Coletar exemplos
hits = []        # corretos no rank 1
near_misses = [] # corretos no rank 2-5
failures = []    # incorretos no top 5

for i, pt_emb in enumerate(pt_embs):
    sims = np.dot(nhe_embs, pt_emb)
    ranked_idx = np.argsort(-sims)
    pos = np.where(ranked_idx == i)[0][0] + 1

    entry = {
        'pt': test_data[i]['pt'],
        'true_nhe': test_data[i]['nhe'],
        'pred_nhe': test_data[ranked_idx[0]]['nhe'],
        'rank': pos,
        'top5_nhe': [test_data[ranked_idx[j]]['nhe'] for j in range(min(5, len(ranked_idx)))],
        'top5_scores': [float(sims[ranked_idx[j]]) for j in range(min(5, len(ranked_idx)))],
        'source': test_data[i].get('source', 'unknown')
    }

    if pos == 1:
        hits.append(entry)
    elif pos <= 5:
        near_misses.append(entry)
    else:
        failures.append(entry)

print(f"\n✅ HITS (correct @1): {len(hits)}")
print(f"⚠️  NEAR-MISSES (@2-5): {len(near_misses)}")
print(f"❌ FAILURES: {len(failures)}")

# ============================================================
# EXEMPLOS DE HITS
# ============================================================
print(f"\n{'='*60}")
print(f"EXEMPLOS DE HITS (corretos no rank 1)")
print(f"{'='*60}")

for i, ex in enumerate(hits[:8]):
    print(f"\n--- Hit {i+1} ---")
    print(f"PT:  {ex['pt'][:120]}...")
    print(f"NHE: {ex['true_nhe'][:120]}...")
    print(f"Fonte: {ex['source'][:50]}")

# ============================================================
# EXEMPLOS DE NEAR-MISSES
# ============================================================
print(f"\n{'='*60}")
print(f"EXEMPLOS DE NEAR-MISSES (corretos no rank 2-5)")
print(f"{'='*60}")

for i, ex in enumerate(near_misses[:5]):
    print(f"\n--- Near-miss {i+1} (rank={ex['rank']}) ---")
    print(f"PT:       {ex['pt'][:100]}...")
    print(f"TRUE NHE: {ex['true_nhe'][:80]}...")
    print(f"PRED NHE: {ex['pred_nhe'][:80]}...")
    print(f"Top 5:    {[n[:30] for n in ex['top5_nhe'][:3]]}")

# ============================================================
# EXEMPLOS DE FAILURES (análise de erros)
# ============================================================
print(f"\n{'='*60}")
print(f"EXEMPLOS DE FAILURES (para análise de erros)")
print(f"{'='*60}")

for i, ex in enumerate(failures[:5]):
    print(f"\n--- Failure {i+1} (rank={ex['rank']}) ---")
    print(f"PT:       {ex['pt'][:100]}...")
    print(f"TRUE NHE: {ex['true_nhe'][:80]}...")
    print(f"PRED NHE: {ex['pred_nhe'][:80]}...")
    print(f"Fonte: {ex['source'][:50]}")

# ============================================================
# ANÁLISE POR FONTE/DOMÍNIO
# ============================================================
print(f"\n{'='*60}")
print(f"ANÁLISE POR DOMÍNIO/FONTE")
print(f"{'='*60}")

# Mapear fontes para domínios
domain_map = {}
for ex in hits + near_misses + failures:
    src = ex['source']
    if 'NAVARRO' in src:
        domain = 'Textbook (Navarro)'
    elif 'CASASNOVAS' in src:
        domain = 'Textbook (Casasnovas)'
    elif 'tycho' in src:
        domain = 'Tycho Corpus'
    elif 'constitution' in src:
        domain = 'Constitution'
    else:
        domain = 'Other'
    ex['domain'] = domain

# Calcular precisão por domínio
domain_stats = defaultdict(lambda: {'total': 0, 'correct': 0})

for ex in hits:
    domain_stats[ex['domain']]['correct'] += 1
    domain_stats[ex['domain']]['total'] += 1

for ex in near_misses + failures:
    domain_stats[ex['domain']]['total'] += 1

print(f"\nPrecisão@1 por domínio:")
for domain, stats in sorted(domain_stats.items(), key=lambda x: -x[1]['correct']/x[1]['total']):
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {domain:25}: {acc:.3f} ({stats['correct']}/{stats['total']})")

# ============================================================
# EXEMPLOS DE CONSTRUÇÕES ESPECÍFICAS
# ============================================================
print(f"\n{'='*60}")
print(f"ANÁLISE DE CONSTRUÇÕES ESPECÍFICAS")
print(f"{'='*60}")

# Frases existenciais com "há"
existential = [ex for ex in failures if ' há ' in ex['pt'] or ex['pt'].startswith('Há ')]
print(f"\nFrases existenciais com 'há' (erros): {len(existential)}")
for ex in existential[:3]:
    print(f"  PT:  {ex['pt'][:80]}")
    print(f"  NHE: {ex['true_nhe'][:80]}")
    print(f"  Pred: {ex['pred_nhe'][:80]}\n")

# Frases curtas (provavelmente do textbook)
short_sentences = [ex for ex in hits if len(ex['pt'].split()) < 8]
print(f"Frases curtas (<8 palavras) corretas: {len(short_sentences)}")
for ex in short_sentences[:3]:
    print(f"  PT: {ex['pt'][:60]}")
    print(f"  NHE: {ex['true_nhe'][:60]}\n")

print(f"\n{'='*60}")
print(f"✅ Avaliação completa!")
print(f"{'='*60}")

Overwriting eval_colab.py


In [ ]:
!python finetune_colab.py


Using device: cuda

Loading corpus from: experiments/02_fasttext/corpus_clean.json
Total pairs: 4997 (bad pairs removed)
Train pairs: 3997
Test pairs:  1000

Loading XLM-R model...
config.json: 100% 615/615 [00:00<00:00, 3.13MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 138kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 4.67MB/s]
tokenizer.json: 9.10MB [00:00, 22.3MB/s]
model.safetensors: 100% 1.12G/1.12G [00:12<00:00, 90.6MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 930.16it/s, Materializing param=pooler.dense.weight]
XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architec

In [ ]:
!python eval_colab.py

Using device: cuda
AVALIAÇÃO XLM-R FINE-TUNED (CORPUS LIMPO)

1. Carregando corpus...
   Total pairs: 4997 (bad pairs removed)
   Test pairs: 1000

2. Carregando modelo fine-tuned...
Loading weights: 100% 199/199 [00:00<00:00, 980.42it/s, Materializing param=pooler.dense.weight] 

3. Computando embeddings...
   PT:  (1000, 768)
   NHE:  (1000, 768)

4. Avaliação quantitativa...

RESULTADOS QUANTITATIVOS
Test pairs: 1000
Precision@1:  0.2470 (247/1000)
Precision@5:  0.5040 (504/1000)
Precision@10: 0.6210 (621/1000)
MRR:          0.3707

ANÁLISE QUALITATIVA

✅ HITS (correct @1): 247
⚠️  NEAR-MISSES (@2-5): 257
❌ FAILURES: 496

EXEMPLOS DE HITS (corretos no rank 1)

--- Hit 1 ---
PT:  Para dirimir conflitos fundiários , o Tribunal de Justiça proporá a criação de varas especializadas , com competência ex...
NHE: Ũbawa arã muramũyasá iwí reséwaára , tribunau justisa yaára umũdu kuri taãmukiriari uka-ita supiaára arãwa , umuyã arã m...
Fonte: constitution

--- Hit 2 ---
PT:  Por onde Maria c

In [ ]:
import shutil
shutil.make_archive("xlmr_finetuned_clean", 'zip', "xlmr_finetuned_clean")

from google.colab import files
files.download("xlmr_finetuned_clean.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!sed -i 's|Path("corpus_merged.json")|Path("experiments/02_fasttext/corpus_merged.json")|' qualitative_eval.py

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

sentences = [
    "Mamé taá aikué siía uiramirĩ?",
    "Ixé aiuseruka Pedro.",
    "Aikué siía igara turusu garapá upé.",
    "Erệ, çuaçú, xa çọ raĩ."
]

for s in sentences:
    tokens = tokenizer.tokenize(s)
    print(f"\n{s}")
    print(f"  Tokens: {tokens}")
    print(f"  Count: {len(tokens)} tokens")


Mamé taá aikué siía uiramirĩ?
  Tokens: ['▁Mam', 'é', '▁ta', 'á', '▁a', 'iku', 'é', '▁si', 'ía', '▁u', 'ira', 'mir', 'ĩ', '?']
  Count: 14 tokens

Ixé aiuseruka Pedro.
  Tokens: ['▁I', 'xé', '▁ai', 'user', 'uka', '▁Pedro', '.']
  Count: 7 tokens

Aikué siía igara turusu garapá upé.
  Tokens: ['▁A', 'iku', 'é', '▁si', 'ía', '▁i', 'gara', '▁tur', 'usu', '▁gara', 'pá', '▁up', 'é', '.']
  Count: 14 tokens

Erệ, çuaçú, xa çọ raĩ.
  Tokens: ['▁Er', 'ệ', ',', '▁', 'çu', 'aç', 'ú', ',', '▁xa', '▁ç', 'ọ', '▁ra', 'ĩ', '.']
  Count: 14 tokens


In [ ]:
import json
import random

with open("experiments/02_fasttext/corpus_clean.json", "r") as f:
    data = json.load(f)

random.seed(42)
random.shuffle(data)
split_idx = int(0.8 * len(data))
test_data = data[split_idx:]

print("As 9 frases PT com 'há' no test set:")
print("="*60)
for i, p in enumerate(test_data):
    if ' há ' in p['pt'] or p['pt'].startswith('Há '):
        print(f"\n{i+1}. PT: {p['pt']}")
        print(f"   NHE: {p['nhe']}")
        print(f"   'aikué' na NHE? {'SIM' if 'aikué' in p['nhe'] else 'NÃO'}")

As 9 frases PT com 'há' no test set:

47. PT: Não , não há homens covardes na cidade .
   NHE: Umbaá , niti aikué apigaua-itá pitua tendaua upé .
   'aikué' na NHE? SIM

107. PT: Há dias frios em Barra ?
   NHE: Aikué será ara-itá irusanga Barra upé ?
   'aikué' na NHE? NÃO

131. PT: Há muito tempo ele é acostumado assim .
   NHE: Akuera iaueuera aé .
   'aikué' na NHE? NÃO

308. PT: Não , não há um homem pobre aqui .
   NHE: Umbaá , niti aikué iepé apigaua pirasua iké .
   'aikué' na NHE? SIM

348. PT: Por aqui há muitas árvores .
   NHE: Kuá rupi aikué siía mirá .
   'aikué' na NHE? SIM

389. PT: Há duas canoas no rio .
   NHE: Aikwé mukũi igara paranã upé .
   'aikué' na NHE? NÃO

513. PT: Há alguns que sabem pescar .
   NHE: Aikué iepé iepé ukuau uaá upinaitika .
   'aikué' na NHE? NÃO

578. PT: Não , não há peixes pretos no rio .
   NHE: Umbaá , niti aikué pirá-itá pixuna paranã upé .
   'aikué' na NHE? SIM

663. PT: Há canoa no igarapé .
   NHE: Aikué igara igarapé upé .
   'aiku

In [ ]:
import json

with open("experiments/02_fasttext/corpus_clean.json", encoding="utf-8") as f:
    data = json.load(f)

# Use the same split as training (80/20, seed 42)
import random
rng = random.Random(42)
rng.shuffle(data)
split_idx = int(0.8 * len(data))
test_data = data[split_idx:]

# Count by source
sources = {}
for p in test_data:
    src = p.get('source', 'unknown')
    sources[src] = sources.get(src, 0) + 1

print("Test set source distribution:")
for src, count in sources.items():
    print(f"  {src}: {count} pairs")

Test set source distribution:
  constitution: 401 pairs
  NAVARRO, Eduardo de Almeida. Curso de Língua Geral (nheengat: 440 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/d0acf29a: 1 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/ab374022: 10 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/a9580875: 7 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/abda0857: 3 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/ca64fa14: 8 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/1d36461f: 10 pairs
  CASASNOVAS, A. Noções de língua geral ou nheengatú: gramátic: 35 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/e454dddd: 2 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/3b7bd381: 4 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/044fc69b: 3 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/5ee7ddb0: 4 pairs
  https://www.tycho.iel.unicamp.br/viewer/translation/8d0c6e8a: 6 pairs
  h

In [ ]:
import os
from datetime import datetime

BACKUP_DIR = f"/content/drive/MyDrive/nheengatu_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"Backup directory: {BACKUP_DIR}")

Backup directory: /content/drive/MyDrive/nheengatu_backup_20260413_014426


In [ ]:
import shutil

# Fine‑tuned model
if os.path.exists("xlmr_finetuned"):
    shutil.copytree("xlmr_finetuned", os.path.join(BACKUP_DIR, "xlmr_finetuned"))

# Corpus and data
for f in ["corpus_merged.json", "corpus_merged.pt", "corpus_merged.nhe"]:
    if os.path.exists(f):
        shutil.copy(f, BACKUP_DIR)

# Experiment scripts and logs
for f in ["finetune_colab.py", "eval_colab.py", "qualitative_eval.py", "finetune.log"]:
    if os.path.exists(f):
        shutil.copy(f, BACKUP_DIR)

# Results folder (if any)
if os.path.exists("results"):
    shutil.copytree("results", os.path.join(BACKUP_DIR, "results"))

print("All files copied to Google Drive.")

All files copied to Google Drive.


In [ ]:
print("\nBackup contents:")
for root, dirs, files in os.walk(BACKUP_DIR):
    level = root.replace(BACKUP_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")


Backup contents:
nheengatu_backup_20260413_014426/
  finetune_colab.py
  eval_colab.py


In [ ]:
import shutil
if os.path.exists("experiments/02_fasttext/corpus_clean.json"):
    shutil.copy("experiments/02_fasttext/corpus_clean.json", BACKUP_DIR)

In [ ]:
import shutil
import os

# Caminho do modelo treinado
model_path = "xlmr_finetuned_clean"
backup_path = "/content/drive/MyDrive/nheengatu_backup_20260413_014426/xlmr_finetuned_clean"

# Verificar se o modelo existe
if os.path.exists(model_path):
    print(f"✅ Modelo encontrado: {model_path}")

    # Copiar para o Drive
    if os.path.exists(backup_path):
        print(f"⚠️ Backup já existe, removendo...")
        shutil.rmtree(backup_path)

    shutil.copytree(model_path, backup_path)
    print(f"✅ Modelo copiado para: {backup_path}")

    # Verificar tamanho
    size = sum(os.path.getsize(os.path.join(dirpath, filename))
               for dirpath, dirnames, filenames in os.walk(model_path)
               for filename in filenames) / (1024**3)
    print(f"📦 Tamanho do modelo: {size:.2f} GB")
else:
    print("❌ Modelo não encontrado!")

✅ Modelo encontrado: xlmr_finetuned_clean
✅ Modelo copiado para: /content/drive/MyDrive/nheengatu_backup_20260413_014426/xlmr_finetuned_clean
📦 Tamanho do modelo: 1.05 GB
